# LAB02 · Dataset público, auditoría y SVM con pipeline completo

**Estado de la aprobación.** El manual de la Unidad 01 exige enviar la comparación de datasets al
docente y **no continuar con el entrenamiento final hasta recibir su aprobación**. Este notebook
ejecuta el entrenamiento sobre el candidato **A · Adult (Census Income)**, elegido tras esa
comparación (ver `docs/ficha_dataset.md`); la aprobación del docente queda **pendiente** y debe
constar antes de entregar.

**Pregunta de investigación.** ¿Puede un SVM con kernel RBF, dentro de un pipeline que imputa,
escala y codifica en one-hot las variables mixtas, superar al baseline y estimar si el ingreso
anual de una persona supera 50.000 dólares, y qué tipo de error es el más costoso en ese uso?

**Unidad de análisis.** Una persona adulta registrada en el censo (un registro = una persona; no es
un evento ni una fila parcial).

**Variable objetivo (target).** `income`: `<=50K` (37.155, 76,07 %) frente a `>50K` (11.687, 23,93 %).

**Error más costoso.** El **falso negativo** (clasificar como `<=50K` a quien realmente supera el
umbral) deja fuera a una persona que califica; si este resultado se usara para decidir crédito,
sería un perjuicio distributivo directo. El falso positivo solo genera contacto innecesario.

**Advertencia.** El dataset es un censo de 1994–1996 de la población activa estadounidense: sus
resultados **no** son extrapolables a la población actual y, en ningún caso, autorizan decisiones
sobre crédito, empleo o seguros. Es un ejercicio de construcción y lectura crítica de un pipeline.

In [1]:
import json
import sys
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score, f1_score, recall_score
from sklearn.model_selection import GridSearchCV, StratifiedKFold

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))

from inf8239_u01.data import DATASET_SPECS, audit_dataframe, audit_table, download_dataset, load_dataset
from inf8239_u01.evaluation import (
    confusion_matrix_frame,
    cv_table,
    plot_confusion,
    plot_cv_results,
    report_frame,
    stratified_split,
)
from inf8239_u01.models import build_preprocessor, build_svm_pipeline

REPORTS = ROOT / "reports"
REPORTS.mkdir(exist_ok=True)
DATASET = "adult"
spec = DATASET_SPECS[DATASET]
print("Dataset:", spec["nombre"], "| licencia:", spec["licencia"])

Dataset: Adult (Census Income) | licencia: CC BY 4.0 (UCI)


## 1. Descarga reproducible

La descarga se hace por script, con validación de URL y registro del SHA-256 de los bytes
obtenidos; no depende de una ruta personal ni de una cuenta.

In [2]:
path = download_dataset(DATASET, ROOT / "data" / "raw")
print("CSV listo:", path)
print((path.parent / f"{path.name}.source.txt").read_text(encoding="utf-8"))
frame = load_dataset(DATASET, path)
print("Forma:", frame.shape)

CSV listo: /home/trek/source/uasd/cienciadedatos/tareas/Unidad_1/INF8239_U01/data/raw/adult.csv
dataset: Adult (Census Income)
url: https://archive.ics.uci.edu/static/public/2/adult.zip
ficha: https://archive.ics.uci.edu/dataset/2/adult
licencia: CC BY 4.0 (UCI)
sha256 del archivo descargado: 7537312dd56c2b98035880805ce99e68183a30ee468aa5329d6df0fbb3cc21bb

Forma: (48842, 15)


## 2. Auditoría del dataset y exclusiones justificadas

Antes de modelar se revisan tipos, ausentes, duplicados, distribución del target y qué columnas
quedan fuera **por motivo semántico**, nunca porque su eliminación mejore la métrica.

In [3]:
audit = audit_dataframe(frame, spec["target"])
print("Filas / columnas:", audit["rows"], "/", audit["columns"])
print("Nulos totales:", audit["total_missing"])
print("Filas duplicadas:", audit["duplicate_rows"])
print("Distribución del target:", audit["target_counts"])
print("Proporción:", {k: round(v, 4) for k, v in audit["target_distribution"].items()})
print()
tabla_auditoria = audit_table(frame)
tabla_auditoria

Filas / columnas: 48842 / 15
Nulos totales: 6465
Filas duplicadas: 52
Distribución del target: {'<=50K': 37155, '>50K': 11687}
Proporción: {'<=50K': 0.7607, '>50K': 0.2393}



,columna,tipo,nulos,porcentaje_nulos,valores_unicos,ejemplo
0,age,int64,0,0.000,74,39
1,workclass,str,2799,5.731,8,State-gov
2,fnlwgt,int64,0,0.000,28523,77516
3,education,str,0,0.000,16,Bachelors
4,education_num,int64,0,0.000,16,13
5,marital_status,str,0,0.000,7,Never-married
6,occupation,str,2809,5.751,14,Adm-clerical
7,relationship,str,0,0.000,6,Not-in-family
8,race,str,0,0.000,5,White
9,sex,str,0,0.000,2,Male


In [4]:
print("Columnas excluidas y motivo:")
for column in spec["exclude"]:
    print(f"  - {column}: {spec['motivo_exclusion']}")

features = frame.drop(columns=spec["exclude"] + [spec["target"]])
target = frame[spec["target"]].astype(str)
print()
print("Variables de entrada:", features.shape[1], "->", list(features.columns))
print("Tipos:", features.dtypes.value_counts().to_dict())
print("Filas conservadas:", len(features), "de", len(frame), "(no se elimina ninguna fila)")

Columnas excluidas y motivo:
  - fnlwgt: fnlwgt es un peso de muestreo del censo, no un atributo de la persona; race, sex y native_country son atributos sensibles que el modelo no necesita para estimar el ingreso y cuyo uso exigiría justificación de equidad
  - race: fnlwgt es un peso de muestreo del censo, no un atributo de la persona; race, sex y native_country son atributos sensibles que el modelo no necesita para estimar el ingreso y cuyo uso exigiría justificación de equidad
  - sex: fnlwgt es un peso de muestreo del censo, no un atributo de la persona; race, sex y native_country son atributos sensibles que el modelo no necesita para estimar el ingreso y cuyo uso exigiría justificación de equidad
  - native_country: fnlwgt es un peso de muestreo del censo, no un atributo de la persona; race, sex y native_country son atributos sensibles que el modelo no necesita para estimar el ingreso y cuyo uso exigiría justificación de equidad

Variables de entrada: 10 -> ['age', 'workclass', 'e

## 3. Partición estratificada 80/20 con semilla 42

La partición ocurre **antes** de cualquier imputación, escalado o codificación: el `ColumnTransformer`
aprende las transformaciones dentro del pipeline, con datos de entrenamiento únicamente.

In [5]:
X_train, X_test, y_train, y_test = stratified_split(features, target)
print("Entrenamiento:", X_train.shape, "| Prueba:", X_test.shape)
print("Balance en entrenamiento:", y_train.value_counts().to_dict())
print("Balance en prueba:", y_test.value_counts().to_dict())
print("Proporción '>50K' en prueba: %.4f" % y_test.value_counts(normalize=True)[" >50K".strip()])

Entrenamiento: (39073, 10) | Prueba: (9769, 10)
Balance en entrenamiento: {'<=50K': 29724, '>50K': 9349}
Balance en prueba: {'<=50K': 7431, '>50K': 2338}
Proporción '>50K' en prueba: 0.2393


## 4. Baseline y SVM con pipeline completo

- **Baseline**: `DummyClassifier(strategy="most_frequent")` dentro del **mismo** preprocesamiento y
  con la **misma** partición, para que la comparación sea justa.
- **SVM**: `Pipeline(ColumnTransformer(SimpleImputer+StandardScaler | SimpleImputer+OneHotEncoder) + SVC(rbf))`.

La búsqueda de hiperparámetros se hace sobre una **submuestra estratificada del 25 %** del
entrenamiento (un ajuste completo del SVM sobre 39.073 filas tarda ~30 s; 45 ajustes serían inviables
en el portátil). El mejor configuración se **reajusta sobre todo el conjunto de entrenamiento** y
solo ese modelo se evalúa sobre la prueba.

In [6]:
from sklearn.base import clone
from sklearn.pipeline import Pipeline

preprocessor = build_preprocessor(X_train)
baseline = Pipeline(steps=[("preprocessor", clone(preprocessor)), ("model", DummyClassifier(strategy="most_frequent"))])
baseline.fit(X_train, y_train)
baseline_pred = baseline.predict(X_test)

submuestra = stratified_split(X_train, y_train, test_size=0.75, random_state=42)
X_search, _, y_search, _ = submuestra
print("Submuestra para la búsqueda:", X_search.shape)

param_grid = {"model__C": [0.1, 1.0, 10.0], "model__gamma": ["scale", 0.01, 0.1]}
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

busqueda = GridSearchCV(
    estimator=build_svm_pipeline(X_search),
    param_grid=param_grid,
    cv=cv,
    scoring="f1_macro",
    n_jobs=-1,
    refit=True,
)
busqueda.fit(X_search, y_search)
tabla_cv = cv_table(busqueda.cv_results_)
tabla_cv.to_csv(REPORTS / "dataset_svm_cv_results.csv", index=False)
print("Mejores parámetros en la submuestra:", busqueda.best_params_, "| F1 macro: %.5f" % busqueda.best_score_)
tabla_cv

Submuestra para la búsqueda: (9768, 10)


Mejores parámetros en la submuestra: {'model__C': 10.0, 'model__gamma': 0.1} | F1 macro: 0.77594


,C,gamma,f1_macro_mean,f1_macro_std,fit_time_mean_s,fit_time_std_s,rank_test_score
0,10.0,0.1,0.775941,0.014134,1.806833,0.121424,1
1,10.0,scale,0.775376,0.013456,2.818932,0.471902,2
2,1.0,scale,0.775268,0.012904,2.735605,0.628623,3
3,1.0,0.1,0.774878,0.013370,2.058249,0.197106,4
4,10.0,0.01,0.772797,0.015065,2.223953,0.391620,5
5,1.0,0.01,0.770157,0.014877,2.019823,0.161484,6
6,0.1,0.1,0.759847,0.014574,2.516162,0.418081,7
7,0.1,scale,0.759203,0.014518,2.661033,0.453396,8
8,0.1,0.01,0.692076,0.018648,2.566907,0.632132,9


In [7]:
plot_cv_results(tabla_cv, REPORTS / "dataset_svm_cv_results.png",
                "Adult · F1 macro en validación por configuración (submuestra 25 %, 5 folds, semilla 42)")
print("Guardado:", REPORTS / "dataset_svm_cv_results.png")

Guardado: /home/trek/source/uasd/cienciadedatos/tareas/Unidad_1/INF8239_U01/reports/dataset_svm_cv_results.png


In [8]:
# Reajuste de la mejor configuración sobre TODO el conjunto de entrenamiento
mejor = build_svm_pipeline(
    X_train,
    C=float(busqueda.best_params_["model__C"]),
    gamma=busqueda.best_params_["model__gamma"],
)
mejor.fit(X_train, y_train)
predicciones = mejor.predict(X_test)

comparativo = pd.DataFrame([
    {
        "modelo": "DummyClassifier (baseline, mismo preprocesamiento)",
        "accuracy": accuracy_score(y_test, baseline_pred),
        "f1_macro": f1_score(y_test, baseline_pred, average="macro"),
        "recall_mas_50k": recall_score(y_test, baseline_pred, pos_label=">50K"),
    },
    {
        "modelo": "SVC rbf con pipeline (C=%s, gamma=%s)" % (
            busqueda.best_params_["model__C"], busqueda.best_params_["model__gamma"]),
        "accuracy": accuracy_score(y_test, predicciones),
        "f1_macro": f1_score(y_test, predicciones, average="macro"),
        "recall_mas_50k": recall_score(y_test, predicciones, pos_label=">50K"),
    },
])
comparativo

,modelo,accuracy,f1_macro,recall_mas_50k
0,"DummyClassifier (baseline, mismo preprocesamie...",0.760672,0.432035,0.000000
1,"SVC rbf con pipeline (C=10.0, gamma=0.1)",0.858839,0.792770,0.614628


## 5. Reporte por clase y matriz de confusión

In [9]:
reporte = report_frame(y_test, predicciones, target_names=["<=50K", ">50K"])
reporte

,clase,precision,recall,f1_score,support
0,<=50K,0.885281,0.935675,0.909781,7431.0
1,>50K,0.750392,0.614628,0.675758,2338.0
2,macro avg,0.817837,0.775151,0.792770,9769.0
3,weighted avg,0.852998,0.858839,0.853773,9769.0


In [10]:
matriz = confusion_matrix_frame(y_test, predicciones, labels=["<=50K", ">50K"], target_names=["<=50K", ">50K"])
print(matriz)
plot_confusion(y_test, predicciones, labels=["<=50K", ">50K"], target_names=["<=50K", ">50K"],
               path=REPORTS / "dataset_svm_confusion_matrix.png",
               title="Adult · SVM con pipeline · matriz de confusión")
print("Guardado:", REPORTS / "dataset_svm_confusion_matrix.png")

            pred_<=50K  pred_>50K
real_<=50K        6953        478
real_>50K          901       1437
Guardado: /home/trek/source/uasd/cienciadedatos/tareas/Unidad_1/INF8239_U01/reports/dataset_svm_confusion_matrix.png


## 6. Análisis de errores sobre una muestra aleatoria con semilla fija

Se seleccionan **20 errores al azar** (semilla 42) y se etiquetan con categorías verificables. La
categoría se propone con reglas transparentes y debe revisarse una a una: la lectura sirve para
plantear la hipótesis del siguiente experimento, no para "corregir" las predicciones.

In [11]:
M = confusion_matrix_frame(y_test, predicciones, labels=["<=50K", ">50K"], target_names=["<=50K", ">50K"]).to_numpy()
tp_bajo, fp_bajo = int(M[0, 0]), int(M[0, 1])   # real <=50K
fn_alto, tn_alto = int(M[1, 0]), int(M[1, 1])   # real >50K

prueba = X_test.copy()
prueba["real"] = y_test.to_numpy()
prueba["predicho"] = predicciones
errores = prueba[prueba["real"] != prueba["predicho"]].reset_index(drop=True)
muestra = errores.sample(n=20, random_state=42).reset_index(drop=True)

def categoria(fila):
    if fila["real"] == "<=50K" and fila["predicho"] == ">50K":
        if fila["capital_gain"] == 0 and fila["hours_per_week"] <= 40:
            return "Falso positivo: umbral de ingresos, sin ganancias de capital y jornada <= 40 h"
        return "Falso positivo: perfil de ingresos bajo con señales de umbral ambiguas"
    if fila["real"] == ">50K" and fila["predicho"] == "<=50K":
        if fila["capital_gain"] > 0:
            return "Falso negativo: ingresos altos con ganancias de capital no capturadas por el perfil"
        if fila["hours_per_week"] >= 50:
            return "Falso negativo: jornada alta que el modelo no asocia con ingreso alto"
        return "Falso negativo: perfil de ingresos altos poco representado en los datos"
    return "Otra"

muestra["categoria"] = muestra.apply(categoria, axis=1)
muestra.to_csv(REPORTS / "dataset_svm_errors.csv", index=False)
muestra[["real", "predicho", "age", "education_num", "hours_per_week", "capital_gain", "capital_loss", "categoria"]]

,real,predicho,age,education_num,hours_per_week,capital_gain,capital_loss,categoria
0,>50K,<=50K,32,13,24,0,0,Falso negativo: perfil de ingresos altos poco ...
1,>50K,<=50K,53,13,21,0,0,Falso negativo: perfil de ingresos altos poco ...
2,>50K,<=50K,47,9,35,0,0,Falso negativo: perfil de ingresos altos poco ...
3,<=50K,>50K,31,14,40,0,1740,"Falso positivo: umbral de ingresos, sin gananc..."
4,>50K,<=50K,74,9,40,0,0,Falso negativo: perfil de ingresos altos poco ...
5,<=50K,>50K,28,13,55,0,0,Falso positivo: perfil de ingresos bajo con se...
6,>50K,<=50K,48,9,48,0,0,Falso negativo: perfil de ingresos altos poco ...
7,<=50K,>50K,52,9,45,0,0,Falso positivo: perfil de ingresos bajo con se...
8,<=50K,>50K,32,13,40,0,0,"Falso positivo: umbral de ingresos, sin gananc..."
9,>50K,<=50K,33,9,40,3103,0,Falso negativo: ingresos altos con ganancias d...


## 7. Métricas finales y artefactos

In [12]:
metricas = {
    "dataset": spec["nombre"],
    "dataset_url": spec["url"],
    "licencia": spec["licencia"],
    "filas_totales": int(len(frame)),
    "filas_entrenamiento": int(len(X_train)),
    "filas_prueba": int(len(X_test)),
    "variables_utilizadas": list(features.columns),
    "columnas_excluidas": spec["exclude"],
    "mejores_parametros": {k: str(v) for k, v in busqueda.best_params_.items()},
    "f1_macro_submuestra_validacion": float(busqueda.best_score_),
    "accuracy": float(accuracy_score(y_test, predicciones)),
    "f1_macro": float(f1_score(y_test, predicciones, average="macro")),
    "recall_mas_50k": float(recall_score(y_test, predicciones, pos_label=">50K")),
    "recall_menos_50k": float(recall_score(y_test, predicciones, pos_label="<=50K")),
    "verdaderos_positivos_mas_50k": tn_alto,
    "falsos_negativos_mas_50k": fn_alto,
    "falsos_positivos_menos_50k": fp_bajo,
    "verdaderos_negativos_menos_50k": tp_bajo,
    "baseline_accuracy": float(accuracy_score(y_test, baseline_pred)),
    "baseline_f1_macro": float(f1_score(y_test, baseline_pred, average="macro")),
    "baseline_recall_mas_50k": float(recall_score(y_test, baseline_pred, pos_label=">50K")),
    "seed": 42,
    "aprobacion_docente": "pendiente",
}
(REPORTS / "dataset_svm_metrics.json").write_text(json.dumps(metricas, indent=2), encoding="utf-8")
joblib.dump(mejor, REPORTS / "dataset_svm_best.joblib")
print(json.dumps(metricas, indent=2))
print("\nGuardado:", REPORTS / "dataset_svm_best.joblib")

{
  "dataset": "Adult (Census Income)",
  "dataset_url": "https://archive.ics.uci.edu/static/public/2/adult.zip",
  "licencia": "CC BY 4.0 (UCI)",
  "filas_totales": 48842,
  "filas_entrenamiento": 39073,
  "filas_prueba": 9769,
  "variables_utilizadas": [
    "age",
    "workclass",
    "education",
    "education_num",
    "marital_status",
    "occupation",
    "relationship",
    "capital_gain",
    "capital_loss",
    "hours_per_week"
  ],
  "columnas_excluidas": [
    "fnlwgt",
    "race",
    "sex",
    "native_country"
  ],
  "mejores_parametros": {
    "model__C": "10.0",
    "model__gamma": "0.1"
  },
  "f1_macro_submuestra_validacion": 0.7759409711368408,
  "accuracy": 0.8588391851776026,
  "f1_macro": 0.7927695595735267,
  "recall_mas_50k": 0.6146278870829769,
  "recall_menos_50k": 0.9356748755214641,
  "verdaderos_positivos_mas_50k": 1437,
  "falsos_negativos_mas_50k": 901,
  "falsos_positivos_menos_50k": 478,
  "verdaderos_negativos_menos_50k": 6953,
  "baseline_accuracy"

## 8. Conclusión

**La comparación de datasets y el pipeline funcionan, pero el resultado exige lectura cuidadosa.** El
baseline `DummyClassifier` alcanza 0,7607 de exactitud y 0,4320 de F1 macro, con **recall 0,0000**
para la clase `>50K`: no es una medida de capacidad predictiva, es la medida del desbalance de
clases. El SVM con pipeline completo sube a **0,8588 de exactitud y 0,7928 de F1 macro**, y recupera
**0,6146** de los casos `>50K` (C=10, gamma=0.1 elegido en validación sobre la submuestra, F1 macro
0,7759). La mejora es real y grande, pero **no** suficiente para el uso que inspira la pregunta: si
el error más costoso es el falso negativo (dejar fuera a quien supera el umbral), un recall de 0,61
significa que **casi cuatro de cada diez personas que califican no se detectan**; en la prueba son
901 falsos negativos sobre 2.338 casos `>50K`, y extrapolando al dataset completo dejarían fuera del
tramo objetivo del orden de 4.500 personas.

La causa está en los datos y no en el modelo: `capital_gain` es cero en el 91,74 % de las filas y solo
el 21,3 % de las personas con `>50K` lo tienen positivo, de modo que la variable que más distingue
los ingresos altos está ausente en cuatro de cada cinco casos positivos; imputar con la mediana no
recrea esa asimetría. A esto se suman 52 filas duplicadas, un censo de 1994–1996 de la población
activa estadounidense (sin año, sin ciclo económico y sin inflación) y una población que no incluye a
menores ni a personas sin ingresos declarados. Los 20 errores revisados apuntan al mismo lugar: los
falsos negativos son personas con estudios superiores y `capital_gain` = 0 cuyo perfil no aparece
en los ejemplos positivos del entrenamiento.

**Decisión:** usar este pipeline como línea base reproducible y auditable, no como herramienta de
decisión sobre personas. El siguiente experimento es (1) medir el desempeño por grupo sin usar los
atributos sensibles que se excluyeron, (2) contrastar con un modelo capaz de representar la
dispersión de `capital_gain` (árboles de decisión o transformación logarítmica) manteniendo la misma
partición y la misma métrica, y (3) repetir el análisis sobre una cohorte más reciente. Antes de
entregar hay que obtener la aprobación del docente sobre el dataset, que en este ejercicio sigue
**pendiente**.